In [2]:
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import DB_FILE

conn = sqlite3.connect(DB_FILE)


First I load the data I need for all Visualizations.

In [6]:
#For the line chart and waterfall / annual P&L summary

annual = pd.read_sql("""
    SELECT
        Calendar.Year,
        COA.Class,
        COA.SubClass,
        SUM(gl.Amount) AS Amount
    FROM GL
    JOIN COA ON GL.Account_key = COA.Account_key
    JOIN Calendar ON GL.Date        = Calendar.Date
    WHERE COA.Report = 'Profit and Loss'
    GROUP BY Calendar.Year, COA.Class, COA.SubClass
""", conn)

#For the bar chart / P&L by region
regional = pd.read_sql("""
    SELECT
        Calendar.Year,
        territory.Region,
        COA.SubClass,
        SUM(gl.Amount) AS Amount
    FROM GL
    JOIN COA ON GL.Account_key = COA.Account_key
    JOIN Calendar ON GL.Date = Calendar.Date
    JOIN territory ON GL.Territory_key = territory.Territory_key
    WHERE COA.Report = 'Profit and Loss'
    GROUP BY Calendar.Year, territory.Region, COA.SubClass
""", conn)

#For the heatmap / monthly expenses
heatmap_data = pd.read_sql("""
    SELECT
        Calendar.Year,
        Calendar.Month,
        COA.SubClass,
        SUM(GL.Amount) AS Amount
    FROM GL
    JOIN COA ON GL.Account_key = COA.Account_key
    JOIN Calendar ON GL.Date = Calendar.Date
    WHERE COA.SubClass IN (
        'Operating Expenses',
        'Depreciation & Amortization',
        'Interest Expense',
        'Taxation'
    )
    GROUP BY Calendar.Year, Calendar.Month, COA.SubClass
""", conn)

print("Data loaded")
print(f"Annual rows:   {len(annual)}")
print(f"Regional rows: {len(regional)}")
print(f"Heatmap rows:  {len(heatmap_data)}")

Data loaded
Annual rows:   30
Regional rows: 90
Heatmap rows:  144


### Line Chart: Revenue vs. Net Profit


In [ ]:
pivot = annual.groupby(['Year', 'SubClass'])['Amount'].sum().unstack('SubClass').fillna(0)
pivot['Revenue']    = pivot['Sales']
pivot['Net Profit'] = (pivot['Sales'] + pivot['Cost of Sales'] +
                       pivot['Operating Expenses'] + pivot['Depreciation & Amortization'] +
                       pivot['Interest Income'] + pivot['Interest Expense'] +
                       pivot['Dividend Income'] + pivot['Exchange Loss/Gain'] +
                       pivot['Gain/Loss on Sales of Asset'] + pivot['Taxation'])

line_df = pivot[['Revenue', 'Net Profit']].reset_index()
line_df = line_df.melt(id_vars='Year', var_name='Metric', value_name='Amount')


fig = px.line(
    line_df,
    x='Year',
    y='Amount',
    color='Metric',
    markers=True,
    title='Revenue vs Net Profit (2018–2020)',
    labels={'Amount': 'Amount (USD)', 'Year': 'Year'},
    color_discrete_map={'Revenue': '#2563EB', 'Net Profit': '#16A34A'}
)

fig.update_layout(
    hovermode='x unified',
    yaxis_tickformat=',.0f',
    legend_title_text=''
)

fig.show()

Note: The gap between Revenue and Net profit is widening - costs are growing faster than than revenue.
<br>
<br>
<br>

### Bar Chart P&L by Region

In [8]:
reg_pivot = regional.groupby(['Year', 'Region', 'SubClass'])['Amount'].sum().reset_index()
reg_pivot = reg_pivot.groupby(['Year', 'Region'])['Amount'].sum().reset_index()
reg_pivot.columns = ['Year', 'Region', 'Net Profit']
reg_pivot['Year'] = reg_pivot['Year'].astype(str)


fig = px.bar(
    reg_pivot,
    x='Region',
    y='Net Profit',
    color='Year',
    barmode='group',
    title='Net Profit by Region (2018–2020)',
    labels={'Net Profit': 'Net Profit (USD)', 'Region': 'Region'},
    color_discrete_sequence=['#93C5FD', '#2563EB', '#1E3A8A']
)

fig.update_layout(
    yaxis_tickformat=',.0f',
    hovermode='x unified',
    legend_title_text='Year'
)

fig.show()

### Heatmap Expense Categories by Month

In [9]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

heatmap_data['Amount'] = heatmap_data['Amount'].abs()

heat_pivot = heatmap_data.groupby(['Month', 'SubClass'])['Amount'].mean().unstack('SubClass').fillna(0)
heat_pivot = heat_pivot.reindex(month_order)


fig = px.imshow(
    heat_pivot.T,
    title='Average Monthly Expenses by Category (2018–2020)',
    labels={'x': 'Month', 'y': 'Expense Category', 'color': 'Amount (USD)'},
    color_continuous_scale='Blues',
    aspect='auto'
)

fig.update_layout(
    xaxis_title='Month',
    yaxis_title='',
    coloraxis_colorbar_tickformat=',.0f'
)

fig.show()

### Waterfall Chart P&L Build

In [10]:
#2020 is the showcase
year = 2020

p = annual[annual['Year'] == year].groupby('SubClass')['Amount'].sum()

# Build waterfall items in P&L order
items = [
    ('Revenue', p.get('Sales', 0)),
    ('Cost of Sales', p.get('Cost of Sales', 0)),
    ('Gross Profit', 0),   # subtotal
    ('Operating Expenses', p.get('Operating Expenses', 0)),
    ('Depreciation & Amort.', p.get('Depreciation & Amortization', 0)),
    ('EBIT', 0),   # subtotal
    ('Interest Income', p.get('Interest Income', 0)),
    ('Interest Expense', p.get('Interest Expense', 0)),
    ('Dividend Income', p.get('Dividend Income', 0)),
    ('Exchange Loss/Gain', p.get('Exchange Loss/Gain', 0)),
    ('Gain/Loss on Asset', p.get('Gain/Loss on Sales of Asset', 0)),
    ('Taxation', p.get('Taxation', 0)),
    ('Net Profit', 0),   # subtotal
]

labels   = [i[0] for i in items]
values   = [i[1] for i in items]
measures = ['absolute' if v == 0 else 'relative' for v in values]

# Subtotals must be 'total' measure
for i, label in enumerate(labels):
    if label in ['Gross Profit', 'EBIT', 'Net Profit']:
        measures[i] = 'total'
        values[i]   = 0


fig = go.Figure(go.Waterfall(
    name=str(year),
    orientation='v',
    measure=measures,
    x=labels,
    y=values,
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    increasing={'marker': {'color': '#16A34A'}},
    decreasing={'marker': {'color': '#DC2626'}},
    totals={'marker': {'color': '#2563EB'}}
))

fig.update_layout(
    title=f'P&L Waterfall — {year}',
    yaxis_tickformat=',.0f',
    showlegend=False,
    yaxis_title='Amount (USD)'
)

fig.show()